_____

<table align="left" width=100%>
    <td>
        <div style="text-align: center;">
          <img src="./images/bar.png" alt="entidades financiadoras"/>
        </div>
    </td>
    <td>
        <p style="text-align: center; font-size:24px;"><b>Introduction to Data Science</b></p>
        <p style="text-align: center; font-size:18px;"><b>Master in Electrical and Computer Engineering</b></p>
        <p style="text-align: center; font-size:14px;"><b>Pedro Cardoso (pcardoso@ualg.pt)</b></p>
    </td>
</table>

_____

__Short Lesson Title:__ Interactive Titanic Dashboard with Panel and Matplotlib

__Summary:__ This lesson guides learners through the creation of an interactive data visualization dashboard using the Panel library, with a focus on the Titanic dataset. Students will learn how to build a responsive dashboard that allows filtering by passenger name, sex, class, and survival status. The lesson covers the use of widgets such as text inputs, buttons, and multi-select dropdowns to dynamically update visualizations. Key plots include a boxplot of passenger ages, a violin plot of age distribution by class and sex, a histogram of age distribution, and a bar plot of survivors by gender and class. The dashboard layout is constructed using Panel’s `Row` and `Column` components, and the visualizations are powered by Matplotlib and Seaborn. Through hands-on coding, students gain experience in data filtering, interactive UI design, and integrating multiple plots into a cohesive analytical tool.

# Titanic Dashboard

This notebook demonstrates how to create a simple dashboard using Panel. The dashboard will display a number of visualizations based on the Titanic dataset. The dataset contains information about passengers on the Titanic, including whether they survived (see the notebook [../3-data-analysis/04_a_exploratory_data_analysis.ipynb](./../3-data-analysis/04_a_exploratory_data_analysis.ipynb), [../3-data-analysis/04_b_exploratory_data_analysis.ipynb](./../3-data-analysis/04_b_exploratory_data_analysis.ipynb) and [./3-data-analysis/04_c_exploratory_data_analysis.ipynb](./../3-data-analysis/04_c_exploratory_data_analysis.ipynb) for more information).

The dashboard will allow users to filter the data based on the:
- Name of the passenger
- Sex of the passenger
- Passenger class
- Whether the passenger survived


The dashboard will contain the following visualizations:

- A boxplot of the age of the passengers
- A violin plot of the age distribution by class
- A histogram of the age distribution
- A bar plot of the number of passengers by class
- Data table with the filtered data

The architecture of the dashboard is as follows:
![Dashboard architecture](./images/layout_and_widgets.png)

The dashboard will look like this:
![Dashboard architecture](./images/titanic_dashboard.png)




So, let's start importing the necessary libraries and loading the data.

In [ ]:
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt

import panel as pn
pn.extension()

from panel.theme import Material
pn.config.design = Material

Next, data is loaded.

In [ ]:
def load_data():
    data = pd.read_excel('./../../3-data-analysis/data/titanic/Titanic.xls')
    return data

df = load_data()

Now, we will create a number of visualizations based on the data. We will create a boxplot of the age of the passengers, a violin plot of the age distribution by class, a histogram of the age distribution, and a bar plot of the number

In [ ]:
def make_age_boxplot(df):
    """ method to create a boxplot of the age of the passengers 
        Args:
            df: pandas DataFrame
        Returns:
            fig: matplotlib figure
    """
    fig, ax = plt.subplots(figsize=(6, 1), tight_layout=True)
    df[['age']].plot.box(ax=ax, 
                         vert=False,
                         color='red')
    ax.set_title('Age', fontsize=5)
    ax.tick_params(labelsize=5)
    ax.set_xlim(0, 90)
    return fig


def make_class_age_sex_violin_plot(df):
    """ method to create a violin plot of the age distribution by class and sex
        Args:
            df: pandas DataFrame
        Returns:
            fig: matplotlib figure
    """
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(5, 2))
    g = sns.violinplot(ax=ax,
                   data=df,
                   x='pclass',
                   y='age',
                   hue='sex')
    # put the legend outside the plot
    g.legend(loc='upper left',  fontsize=5)
    ax.set_title('Age distribution by class', fontsize=8)
    ax.set_xlabel('Passenger Class', fontsize=7)
    ax.set_ylabel('Age', fontsize=7)
    ax.tick_params(labelsize=5)
    
    return fig
    
def make_ages_histogram(df):
    """ method to create a histogram of the age distribution
        Args:
            df: pandas DataFrame
        Returns:
            fig: matplotlib figure
    """
    fig, ax = plt.subplots(figsize=(5, 2))
    df['age'].plot(kind='hist',
                   ax=ax,
                   alpha=0.1,
                   bins=20,
                   color='green')
    ax.set_xlabel('Age', fontsize=7)
    ax.set_ylabel('Count', fontsize=7)
    ax.set_title('Age distribution', fontsize=8)
    ax.tick_params(labelsize=5)
    return fig

def make_survivors_by_gender_and_class(df):
    """ method to create a bar plot of the number of survivors by gender and class
        Args:
            df: pandas DataFrame
        Returns:
            fig: matplotlib figure
    """
    fig, ax = plt.subplots(figsize=(5, 2))
    g = df.pivot_table(index='sex',
                       values='survived', 
                       columns='pclass', 
                       aggfunc='sum').plot(kind='bar', ax=ax)
    g.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=7)
    ax.set_title('Survivors by gender and class', fontsize=7)
    ax.set_ylabel('Number of survivors', fontsize=7)
    ax.tick_params(labelsize=7)
    return fig

# uncomment to test and see the plots
# make_age_boxplot(df)
# make_class_age_sex_violin_plot(df)
# make_ages_histogram(df)
# make_survivors_by_gender_and_class(df)


With the plots created, we can now create the dashboard. 

1. In this it's used the `pn.pane.Matplotlib` to display the plots. Its arguments are the call to the function that creates and returns the plot and the width of the plot.
2. The `pn.widgets.DataFrame` is used to display the data table. Its arguments are the DataFrame and the width and height of the table.
3. The `pn.layout.Column` and `pn.layout.Row` are used to display the plots and the table in a column and row.


In [ ]:
age_boxplot_panel = pn.pane.Matplotlib(make_age_boxplot(df), width=1200)
passenger_by_class_panel = pn.pane.Matplotlib(make_class_age_sex_violin_plot(df), width=400)
ages_hist_panel = pn.pane.Matplotlib(make_ages_histogram(df), width=400)
survivors_by_gender_and_class_panel = pn.pane.Matplotlib(make_survivors_by_gender_and_class(df), width=400)


df_panel = pn.widgets.DataFrame(df, width=1200, height=400)

data_visualization_panel = pn.layout.Column(
    age_boxplot_panel,
    pn.layout.Row(
        passenger_by_class_panel,
        ages_hist_panel,
       ),
    survivors_by_gender_and_class_panel,
    df_panel
)

Next, the interactive widgets are created, namely:
1. A text input to filter the data by name and button to apply the text filter
1. A multi-select to filter the data by sex
2. A multi-select to filter the data by passenger class
3. A multi-select to filter the data by survival status
4. A panel to hold the filter widgets


In [ ]:
# Text filter on the passenger name & Button to apply the text filter
filter_text_intput = pn.widgets.TextInput(name='Filter Name', value='')
filter_button = pn.widgets.Button(name='Apply Filter', button_type='primary')

# MultiSelect to filter on sex
filter_sex = pn.widgets.MultiSelect(name='Sex', 
                                    options=['male', 'female'], 
                                    value=['male', 'female'])

# MultiSelect to filter on passenger class
filter_passenger_class = pn.widgets.MultiSelect(name='Passenger Class', 
                                                options=['1', '2', '3'],
                                                value=['1', '2', '3'])

# MultiSelect to filter if passenger survived
filter_survived = pn.widgets.MultiSelect(name='Survived', 
                                         options=[0, 1],
                                         value=[0, 1])

# Panel to hold the filter widgets
filter_panel = pn.layout.Column(
    pn.layout.Column(
        filter_text_intput,
        filter_button),
    pn.layout.Column(
        filter_sex,
        filter_passenger_class,
        filter_survived),
    styles=dict(background='WhiteSmoke')
)

Next we need to create the `update_indicator` function that will be used to update the plots and the table when the widgets are changed, more precisely, when an event is triggered.

In [ ]:
def update_indicator(event):
    text_widgetvalue = filter_text_intput.value
    sex_widget_values = filter_sex.value # list of selected values
    passenger_class_widget_values = filter_passenger_class.value # list of selected values
    survived_widget_values = filter_survived.value # list of selected values

    # build filter based on widget values - start with all True and then incrementally "add" the filters    
    filter = pd.Series([True] * len(df))

    # filter on name if it is not empty
    if text_widgetvalue:
        filter = filter & df['name'].str.contains(text_widgetvalue, case=False)
    
    # filter on sex if it is not empty
    if sex_widget_values:
        filter = filter & df['sex'].isin(sex_widget_values)
    
    # filter on passenger class if it is not empty
    if passenger_class_widget_values:
        passenger_class_widget_values_as_int = [int(x) for x in passenger_class_widget_values]
        filter = filter & df['pclass'].isin(passenger_class_widget_values_as_int)  
        
    # filter on survived if it is not empty
    if survived_widget_values:
        survived_widget_values_as_int = [int(x) for x in survived_widget_values]
        filter = filter & df['survived'].isin(survived_widget_values_as_int)
    
    # update the DataFrame panel    
    df_panel.value = df[filter] 
    
    # update the boxplot
    age_boxplot_panel.object = make_age_boxplot(df[filter])
    
    # update the violin plot
    passenger_by_class_panel.object = make_class_age_sex_violin_plot(df[filter])
    
    # update the histogram
    ages_hist_panel.object = make_ages_histogram(df[filter])
    
    # update the survivors by gender and class plot
    survivors_by_gender_and_class_panel.object = make_survivors_by_gender_and_class(df[filter])

Finally, the `pn.bind` function is used to bind the event to the function. Its arguments are the function to call when the event is triggered and the event to bind.

In [ ]:
pn.bind(update_indicator, filter_button, watch=True)
pn.bind(update_indicator, filter_sex, watch=True)
pn.bind(update_indicator, filter_passenger_class, watch=True)
pn.bind(update_indicator, filter_survived, watch=True)

In [ ]:
pn.layout.Row(
    filter_panel,
    data_visualization_panel,
).servable()